# AgriNav -- Phase-2 Detector Training on curated RICE (Colab / GPU)

**Phase 2.** Fine-tunes the `weeddet_v6b` detector on the **curated RICE** dataset,
warm-started from the **phase-1 RiceSEG backbone**. This notebook only *drives*
`agrinav.training.weeddet_train` from the repo -- it defines no model, loss, or
training logic of its own.

**This is a TWO-class task:** `rice_protect` (COCO id 1), `weed_target` (COCO id 2).
The 3-class RiceSEG detector config does **not** apply; we use
`configs/training/detector_rice_phase2.yaml`.

**Which data, and why this split.** Images/annotations come from the *curated* RICE
build (`agrinav_intake_2026-07-21/deliverable/detection/RICE/`): a grouped,
leakage-free split (capture-series families, 40-frame contiguous blocks,
weed-balanced 70/20/10), with per-image filter decisions and classes already remapped
to the project taxonomy. **Do not substitute the Roboflow-native split inside
`RICE.coco.zip`** -- its train/valid/test share video-frame families (adjacent frames
land on both sides), so any val metric computed on it is inflated.

**Still exploratory.** The deliverable is **loss convergence + qualitative val
predictions**, NOT a defensible mAP. The COCO evaluator exists (`agrinav evaluate`),
but the model-runner adapter and the canonical decode (Gate-4 P1-6) are not done, so
no mAP cell is included yet -- by design, rather than producing a number that would
change once the decode is unified.

**The test split is sealed** and is not even uploaded: `RICE_curated_phase2.zip`
contains only `train` and `valid`.

**Before you run:** `Runtime -> Change runtime type -> GPU`, and in Drive:
- `MyDrive/agrinav_data/rice_phase2/RICE_curated_phase2.zip` -- images + annotations
- `MyDrive/agrinav_data/out/riceseg_backbone.pth` -- the phase-1 backbone


## 1. Confirm GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code (clone the private repo)

**This repository is PRIVATE**, so an anonymous clone fails with
`could not read Username for 'https://github.com'` -- Colab has no interactive prompt.

Pick ONE of:

1. **Colab secret (recommended).** Left sidebar -> Secrets -> add `GITHUB_TOKEN` with a
   fine-grained GitHub PAT (**Contents: Read**) and toggle notebook access on. The token
   is never printed, never saved into the notebook, and is stripped from the git remote
   after cloning.
2. **Make the repo public** -- then this cell works with no token.
3. **Skip GitHub:** upload the repo folder to Drive and set `REPO_DIR` to it.

The cell **stops immediately** if the code isn't available, so later cells can't cascade.


In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/Bmerrysmith/Autonomous-tractor-system.git'
# Pinned checkout. A mutable branch name means two runs of "the same" notebook can
# execute different code -- and it silently did: the phase-2 runs of 2026-07-28
# cloned `master`, which had no --val-ann-file, so they selected checkpoints on
# TRAINING loss while the notebook's own comments claimed otherwise.
# Set REPO_SHA = None to deliberately take the tip of REPO_REF instead.
REPO_REF  = 'feat/training-observability'
REPO_SHA  = '7c6271fbb1e846dd009503ab58017b7601350b0a'
REPO_DIR  = '/content/agrinav'

# Capabilities this notebook requires from the checkout. Each is a CLI flag that
# must exist in the trainer; a stale clone fails here instead of halfway through
# an 18-epoch run.
REQUIRED_FLAGS = ('riceseg-backbone', 'val-ann-file', 'val-images-root', 'bn-policy')

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
print('GitHub token found in Colab secrets:', bool(token))

def _redact(s):
    return s.replace(token, '***') if token else s

# GIT_TERMINAL_PROMPT=0 turns an auth failure into an immediate error, not a hang.
env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL

def _git(*args, check=True):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], env=env, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f'git {args[0]} failed:\n' + _redact(r.stderr))
    return r.stdout.strip()

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '--branch', REPO_REF, auth_url, REPO_DIR],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            'CLONE FAILED -- stopping so later cells do not cascade.\n\n'
            + _redact(r.stderr) +
            '\nFix: add a GITHUB_TOKEN Colab secret (fine-grained PAT, Contents:Read), '
            'or make the repo public, or upload the repo to Drive and set REPO_DIR.')
    _git('remote', 'set-url', 'origin', REPO_URL)
else:
    _git('remote', 'set-url', 'origin', auth_url)
    _git('fetch', 'origin', '--prune')
    _git('remote', 'set-url', 'origin', REPO_URL)

if REPO_SHA:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--depth', '50', auth_url, REPO_SHA],
                   env=env, capture_output=True, text=True)
    _git('checkout', '--force', REPO_SHA)
else:
    _git('checkout', '--force', f'origin/{REPO_REF}')

os.chdir(REPO_DIR)
HEAD_SHA = _git('rev-parse', 'HEAD')
if REPO_SHA and HEAD_SHA != REPO_SHA:
    raise SystemExit(f'checkout landed on {HEAD_SHA}, expected {REPO_SHA}')

trainer = 'src/agrinav/training/weeddet_train.py'
if not os.path.exists(trainer):
    raise SystemExit(f'checkout at {HEAD_SHA} has no {trainer} -- wrong ref?')
with open(trainer, encoding='utf-8') as fh:
    trainer_src = fh.read()
absent = [flag for flag in REQUIRED_FLAGS if flag not in trainer_src]
if absent:
    raise SystemExit(
        f'this checkout ({HEAD_SHA[:8]}) is missing required trainer flag(s): {absent}.\n'
        'Delete /content/agrinav and re-run, or update REPO_REF/REPO_SHA to a commit that '
        'has them. Running anyway would silently change what the run measures.')

print(f'repo ready at {REPO_DIR} @ {HEAD_SHA}')
print('  required flags present:', ', '.join(REQUIRED_FLAGS))


## 4. Install the package

Colab already ships a CUDA torch/torchvision satisfying the `train` extra's range, so pip
keeps it and only adds the rest.

In [ ]:
import subprocess

# check=True: a failed install must stop the notebook. `get_ipython().system(...)`
# discards the exit code, so a broken environment used to sail on into training.
subprocess.run(['pip', 'install', '-q', '-e', '.[train]'], check=True)
import agrinav
print('agrinav', agrinav.__version__, 'installed from', os.getcwd())


## 5. Extract the curated RICE data + locate the backbone

Extracts `RICE_curated_phase2.zip` to fast local Colab disk (`/content/rice_curated`),
**not** Drive -- training reads thousands of small files and the Drive mount is far too
slow for that.

The archive holds `images/{train,valid}/` and `annotations/instances_{train,valid}.coco.json`.
**It contains no `test`**, so the sealed split cannot be touched even by accident; the cell
asserts that rather than trusting it.

In [ ]:
import glob, hashlib, json, zipfile
from pathlib import PurePosixPath

DRIVE_P2 = '/content/drive/MyDrive/agrinav_data/rice_phase2'
LOCAL    = '/content/rice_curated'

# Name of the rebuilt archive. The 2026-07-27 `RICE_curated_phase2.zip` is
# CONTAMINATED -- it mis-exported 231 of the 261 intended sealed-test images into
# train/valid and omitted 233 intended train/valid images (it dropped the native
# test/ folder instead of applying grouped_split.json). Do not train on it.
ARCHIVE = 'RICE_phase2_rebuild.zip'
BANNED_ARCHIVES = {'RICE_curated_phase2.zip'}

zip_path = f'{DRIVE_P2}/{ARCHIVE}'
if not os.path.exists(zip_path):
    hits = [p for p in glob.glob(f'/content/drive/MyDrive/**/{ARCHIVE}', recursive=True)]
    assert hits, (f'{ARCHIVE} not found in Drive. Build it with '
                  '`agrinav data-build-rice-phase2 build --source-root <deliverable>/detection/RICE '
                  '--out-root <dir> --legacy-archive <old zip>` then `... package --out-root <dir> '
                  f'--zip {ARCHIVE}`, and upload it to {DRIVE_P2}/.')
    zip_path = hits[0]
assert os.path.basename(zip_path) not in BANNED_ARCHIVES, (
    f'{os.path.basename(zip_path)} is the contaminated archive. See the VOID.md files in '
    f'{DRIVE_P2}/runs/.')

def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

ARCHIVE_SHA256 = _sha256(zip_path)
print('zip   :', zip_path)
print('sha256:', ARCHIVE_SHA256)

def _safe_members(names):
    """Reject traversal/absolute paths and any member under a `test` directory.

    The old check was `'/test/' in member`, which a member named `test/x.jpg`
    (no leading slash) walks straight past, and it said nothing about `..` or
    absolute paths before `extractall`.
    """
    for name in names:
        if name.endswith('/'):
            continue
        parts = PurePosixPath(name).parts
        assert not PurePosixPath(name).is_absolute() and not name.startswith('/'), \
            f'absolute path in archive: {name}'
        assert '..' not in parts, f'path traversal in archive: {name}'
        assert 'test' not in parts, (
            f'archive contains a test-split member ({name}). The test split must stay '
            'physically out of any archive used for training.')
    return True

if not os.path.isdir(f'{LOCAL}/images/train'):
    os.makedirs(LOCAL, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        _safe_members(z.namelist())
        z.extractall(LOCAL)
else:
    print('reusing existing extraction at', LOCAL,
          '-- delete it if the archive changed')

TRAIN_JSON = f'{LOCAL}/annotations/instances_train.coco.json'
VAL_JSON   = f'{LOCAL}/annotations/instances_valid.coco.json'
TRAIN_IMGS = f'{LOCAL}/images/train'
VAL_IMGS   = f'{LOCAL}/images/valid'
for p in (TRAIN_JSON, VAL_JSON, TRAIN_IMGS, VAL_IMGS):
    assert os.path.exists(p), f'missing after extract: {p}'

# Hash every emitted annotation file: the run manifest records these, so a future
# reader can tell whether two runs saw the same labels.
JSON_SHA256 = {os.path.basename(p): _sha256(p) for p in (TRAIN_JSON, VAL_JSON)}

# Integrity gate: re-verify the extracted tree from disk (per-image hashes,
# decoded dimensions vs COCO records, in-bounds boxes, no cross-split duplicates,
# no stray files). Fails closed.
subprocess.run(
    ['python', '-B', '-u', '-m', 'agrinav.data.build_rice_phase2', 'preflight',
     '--out-root', LOCAL],
    check=True)

for name, j in (('train', TRAIN_JSON), ('valid', VAL_JSON)):
    d = json.load(open(j))
    cats = {c['id']: c['name'] for c in d['categories']}
    print(f'{name}: {len(d["images"])} imgs  {len(d["annotations"])} anns  cats={cats}')

BACKBONE = '/content/drive/MyDrive/agrinav_data/out/riceseg_backbone.pth'
assert os.path.exists(BACKBONE), (
    f'phase-1 backbone not found at {BACKBONE}. Run the RiceSEG pretraining notebook '
    'first, or set BACKBONE=None to warm-start from ImageNet instead.')
print('backbone:', BACKBONE)


## 6. Sanity gate: self-test

One forward+backward on synthetic tensors (including a zero-GT image); no data, no network.
Must print PASS before spending GPU time.

In [ ]:
# check=True: the self-test is a gate. A nonzero exit must stop the notebook here,
# not be printed and ignored.
subprocess.run(
    ['python', '-B', '-u', '-m', 'agrinav.training.weeddet_train', '--self-test'],
    check=True)
print('self-test gate: PASSED')


## 7. Wiring gate: overfit 8 real RICE images

Cheap insurance that the *data* path is right, not just the model path. If the detector
cannot drive the loss down on 8 images it certainly will not learn on 1798 -- and you find
out in a minute rather than an hour. Exits non-zero if the loss fails to decrease.

In [ ]:
# Wiring gate: overfit 8 real RICE images. check=True, so a failure stops the
# notebook before the full run.
#
# Caveat this gate does NOT cover, by construction: it runs with
# --no-pretrained-backbone (so not the RiceSEG warm-start path used below), and it
# passes on `final_loss < initial_loss` alone -- no decode, no ranking, no AP.
# Treat it as "the loss plumbing is connected", not "the detector works".
subprocess.run(
    ['python', '-B', '-u', '-m', 'agrinav.training.weeddet_train',
     '--ann-file', TRAIN_JSON, '--images-root', TRAIN_IMGS,
     '--class-names', 'rice_protect,weed_target',
     '--overfit', '8', '--batch-size', '2', '--img-size', '512',
     '--no-pretrained-backbone'],
    check=True)
print('overfit gate: PASSED (loss direction only)')


## 8. Train (exploratory, RiceSEG-warm-started)

Uses `configs/training/detector_rice_phase2.yaml` (2-class, 512px, batch 8, AMP, ~18
epochs) with `--riceseg-backbone`, which loads the phase-1 backbone **instead of**
ImageNet. That loader **fails closed**: every expected `backbone.*` tensor must match by
name and shape or it raises -- a silent partial load cannot happen.

`weeddet_best.pth` is selected by lowest **train** loss; there is no val-based selection
yet (that needs the evaluator adapter), so treat the exported checkpoint as exploratory.

In [ ]:
import datetime, hashlib

TS      = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f'{DRIVE_P2}/runs/weeddet_rice_{TS}'
os.makedirs(RUN_DIR, exist_ok=True)

CONFIG      = 'configs/training/detector_rice_phase2.yaml'
CLASS_NAMES = 'rice_protect,weed_target'
SEED        = 42

def _git(*a):
    try:
        return subprocess.check_output(['git', '-C', '.', *a], text=True).strip()
    except Exception:
        return None

def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

# Dataset provenance from the extracted tree, when the archive carries it.
dataset_provenance = {}
_prov = f'{LOCAL}/manifests/provenance.json'
if os.path.exists(_prov):
    _p = json.load(open(_prov))
    dataset_provenance = {
        'built_by': _p.get('built_by'),
        'split_manifest_sha256': _p.get('split_manifest_sha256'),
        'grouping_method': _p.get('grouping_method'),
        'exif_policy': _p.get('exif_policy'),
        'images_reoriented': _p.get('images_reoriented'),
        'test_split_status': _p.get('test_split_status'),
        'per_split': _p.get('per_split'),
    }

manifest = {
    'run_id': f'weeddet_rice_{TS}',
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'phase': 'phase-2 detector, curated RICE, RiceSEG-warm-started',
    'git_commit': _git('rev-parse', 'HEAD'),
    'git_ref_requested': REPO_SHA or REPO_REF,
    'git_branch': _git('rev-parse', '--abbrev-ref', 'HEAD'),
    'git_dirty': bool(_git('status', '--porcelain')),
    'config_file': CONFIG,
    'config_sha256': _sha256(CONFIG),
    'class_map': {n: i for i, n in enumerate(CLASS_NAMES.split(','))},
    'seed': SEED,
    'train_split': TRAIN_JSON,
    'val_split': VAL_JSON,
    'dataset_archive': zip_path,
    'dataset_archive_sha256': ARCHIVE_SHA256,
    'annotation_sha256': JSON_SHA256,
    'dataset_provenance': dataset_provenance,
    'backbone_init': BACKBONE,
    'backbone_sha256': _sha256(BACKBONE),
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'selection_metric': ('validation COCO AP of the EMA weights when val_ap_interval > 0 in the config, else validation loss'),
    'note': ('Exploratory. Decode is class-aware and the model-to-COCO adapter is wired, '
             'so per-epoch validation AP is real and selects the checkpoint. What is still '
             'missing for a defensible headline number: a replacement grouped test split '
             '(the manifest one is burned) and same-protocol baselines. The test split is '
             'physically absent from this archive -- verified member-by-member in the '
             'extraction cell, not assumed.'),
}
with open(f'{RUN_DIR}/run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, sort_keys=True)
print('run dir:', RUN_DIR)
print(json.dumps(manifest, indent=2, sort_keys=True))


In [ ]:
# Detached (`nohup ... &`) on purpose:
#   * a browser/frontend disconnect can no longer take the training process with it;
#   * the per-epoch summaries land in a file on Drive instead of only in this
#     cell's output. Colab truncates long cell output ("Streaming output
#     truncated to the last 5000 lines"), which is how epochs 1-7 of an earlier
#     run were lost permanently.
# stdout -> train.log on Drive (small: one line per epoch).
# stderr -> /content/progress.log (huge: the tqdm bars; local disk, disposable).
#
# --val-ann-file/--val-images-root add held-out validation each epoch. With
# val_ap_interval > 0 in the config, the EMA weights are decoded over the whole
# val split and scored with pycocotools every N epochs, and COCO AP selects
# `best` -- replacing both training loss and val loss as the selection metric.
#
# Detaching means this cell cannot report the trainer's exit code, so it verifies
# the launch instead: the process must still be alive a few seconds later. The
# gates above run with check=True, so a failed self-test or overfit stops the
# notebook before reaching this cell.
import shlex, time

argv = ['python', '-B', '-u', '-m', 'agrinav.training.weeddet_train',
        '--ann-file', TRAIN_JSON, '--images-root', TRAIN_IMGS,
        '--val-ann-file', VAL_JSON, '--val-images-root', VAL_IMGS,
        '--config', CONFIG, '--riceseg-backbone', BACKBONE,
        '--seed', str(SEED), '--checkpoint-dir', RUN_DIR]
with open(f'{RUN_DIR}/launch_command.txt', 'w') as f:
    f.write(shlex.join(argv) + '\n')

train_log = f'{RUN_DIR}/train.log'
with open(train_log, 'wb') as out, open('/content/progress.log', 'wb') as err:
    proc = subprocess.Popen(argv, stdout=out, stderr=err, start_new_session=True)

time.sleep(20)
if proc.poll() is not None:
    print(open(train_log).read()[-4000:])
    raise SystemExit(
        f'training exited immediately with code {proc.returncode}. See {train_log} '
        'and /content/progress.log.')

print(f'training started detached, pid {proc.pid}')
print(f'  epoch summaries : {train_log}')
print( '  progress bars   : /content/progress.log')
print(f'  launch command  : {RUN_DIR}/launch_command.txt')
print('Run the next cell to poll.')


### 8a. Poll training

Re-run this while training runs. It reads the log files, so it never floods the notebook.


In [ ]:
# Poll progress. Safe to re-run as often as you like; it prints a bounded amount
# of text, so the notebook stays responsive no matter how long training runs.
import subprocess, time

print(subprocess.run(['tail', '-n', '25', f'{RUN_DIR}/train.log'],
                     capture_output=True, text=True).stdout)
print('--- latest progress-bar line ---')
print(subprocess.run(f"tail -c 2000 /content/progress.log | tr '\\r' '\\n' | tail -n 1",
                     shell=True, capture_output=True, text=True).stdout)
print('--- still running? ---')
running = subprocess.run(['pgrep', '-f', 'agrinav.training.weeddet_train'],
                         capture_output=True, text=True).stdout.strip()
print('yes, pid(s): ' + running if running else 'no -- finished or stopped')


### 8b. Confirm completion and read the loss curve

Run once training reports it is no longer running.


In [ ]:
# Did the run actually finish? `status.json` answers this unambiguously.
#
# Before this existed, `save_every: 4` against `num_epochs: 18` meant a
# *completed* run wrote no epoch-18 checkpoint -- its newest periodic file was
# weeddet_epoch16.pth, identical on disk to a run killed at 16. Four finished
# runs were misread as crashes that way.
import json

with open(f'{RUN_DIR}/status.json') as fh:
    status = json.load(fh)
print(json.dumps(status, indent=2))
assert status['completed'], (
    f"run stopped at epoch {status['epochs_completed']}/{status['epochs_planned']}; "
    f"weeddet_last.pth holds that epoch and metrics.jsonl has every finished epoch.")

# The full loss curve, straight off disk -- no scrolling, nothing truncated.
rows = [json.loads(line) for line in open(f'{RUN_DIR}/metrics.jsonl') if line.strip()]
print(f"\n{'ep':>3}  {'train':>8}  {'val_raw':>8}  {'val_ema':>8}  {'lr':>9}")
for row in rows:
    print(f"{row['epoch']:>3}  {row['train/total_loss']:>8.4f}  "
          f"{row.get('val_raw/total_loss', float('nan')):>8.4f}  "
          f"{row.get('val_ema/total_loss', float('nan')):>8.4f}  {row['lr']:>9.6f}")
print(f"\nbest epoch {status['best_epoch']} by {status['best_metric_name']} "
      f"= {status['best_metric_value']:.4f}")


## 9. Qualitative predictions on VALIDATION images

Loads the best (EMA) checkpoint and draws predicted boxes on ~6 **validation** images. This
is the exploratory deliverable, **not** a mAP score. Boxes carry the class **name** and score
(never colour alone). The **test split is never touched** -- it is not even present.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from agrinav.training.weeddet_train import (
    _CocoSplitDataset, load_checkpoint_model, predict_image)

device      = 'cuda' if torch.cuda.is_available() else 'cpu'
class_names = CLASS_NAMES.split(',')
ckpt        = f'{RUN_DIR}/weeddet_best.pth'
assert os.path.exists(ckpt), f'no checkpoint at {ckpt} -- did training finish?'
model = load_checkpoint_model(ckpt, num_classes=len(class_names), device=device)

val_ds = _CocoSplitDataset(VAL_JSON, VAL_IMGS, tuple(class_names), img_size=512)
items  = val_ds.items()
random.Random(0).shuffle(items)
picks  = items[:6]

colors = ['lime', 'red']
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, (img_id, path, W, H) in zip(axes.ravel(), picks):
    img, boxes, scores, labels = predict_image(
        model, path, img_size=512, device=device, score_thr=0.3)
    ax.imshow(img); ax.set_axis_off()
    ax.set_title(f'{os.path.basename(path)[:28]}  ({len(boxes)} dets @0.3)', fontsize=9)
    for (x1, y1, x2, y2), s, l in zip(boxes, scores, labels):
        c = colors[int(l) % len(colors)]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor=c, linewidth=2))
        ax.text(x1, max(y1 - 4, 0), f'{class_names[int(l)]} {s:.2f}', color=c, fontsize=8,
                bbox=dict(facecolor='black', alpha=0.5, pad=1, edgecolor='none'))
plt.suptitle('Phase-2 RICE exploratory predictions on VAL (no mAP yet; test sealed)',
             fontsize=13)
plt.tight_layout(); plt.show()

## Next steps

1. Read the per-epoch `avg_loss` stream -- the goal is a **smoothly decreasing** loss with
   no NaNs, plus qualitatively sensible boxes on VAL.
2. Artifacts land in the Drive run dir: `weeddet_best.pth`, periodic `weeddet_epochN.pth`,
   and `run_manifest.json` (git commit + dirty flag + config + class map + seed +
   **backbone sha256**, so the run is traceable to the exact phase-1 backbone).
3. **The ImageNet-vs-RiceSEG comparison is one flag:** re-run cell 8 *without*
   `--riceseg-backbone` for the ImageNet control, same seed and config. That is the
   experiment phase-1 exists to justify -- but it is only a fair comparison once
   **val-based selection and mAP** exist, since train loss alone cannot rank two backbones.
4. **Out of scope here:** a calibrated operating point, COCO mAP, and the sealed test-set
   evaluation. Those need Gate-4 P1-6 (one canonical decode) plus the model-runner adapter
   feeding `agrinav evaluate`.